# Admin API로 조직 관리

팀을 Claude API에 온보딩할 때마다 Console에서 똑같이 클릭해야 합니다. 각 사람을 초대하고, workspace를 만들고, member를 추가하고, 어떤 API key가 있는지 확인하고, CI의 identity를 설정합니다. 수동으로 하면 기록이 남지 않고 팀 몇 개 이상으로 확장하기 어렵습니다.

Admin API는 이 제어 기능을 `/v1/organizations` 아래 REST endpoint로 제공하며, Python SDK는 이를 `client.beta.organization`으로 감쌉니다. 이 notebook은 팀을 처음부터 끝까지 온보딩하면서 조직의 API key를 감사한 다음 생성한 모든 것을 제거합니다.

**이 notebook을 마치면 다음을 할 수 있습니다.**

- OAuth login 또는 Console Admin API key로 조직 admin 인증
- 사용자 초대, 조직 role 설정, member가 있는 workspace 생성
- 조직의 API key 감사 및 rate limit 읽기
- automation용 service account 생성 및 workspace 접근 권한 부여

동일한 호출은 provisioning script나 예약된 감사 작업에서도 변경 없이 작동합니다.

## 사전 요구 사항

- Python 기본 지식과 Claude Console 조직, workspace, API key에 대한 지식
- Python 3.11+ 및 `client.beta.organization`이 처음 포함된 release인 `anthropic>=1.1.0`
- `admin`, `owner`, `primary_owner` role을 보유한 Claude Console **조직**. 개인 account에서는 Admin API를 사용할 수 없습니다.
- 선택 사항: OAuth login 경로용 [`ant` CLI](https://platform.claude.com/docs/en/cli-sdks-libraries/cli/authentication)

## Admin으로 인증

하나의 workspace로 범위가 지정된 API key는 Admin API를 호출할 수 없습니다. 세 가지 credential을 허용합니다. `org:admin` OAuth token, Admin API key, 하나의 workspace로 범위가 지정되지 않은 개인 또는 service account key입니다. 이 notebook은 처음 두 가지를 사용하며, 둘 다 조직 admin만 만들 수 있습니다.

| Credential | 얻는 방법 | 범위 |
|---|---|---|
| `org:admin` scope가 있는 OAuth token | `ant auth login --profile admin --scope org:admin` | 이 notebook의 모든 endpoint |
| Admin API key (`sk-ant-admin01-...`) | Claude Console → Settings → [Admin keys](https://platform.claude.com/settings/admin-keys) | service account와 workload identity federation을 제외한 모든 것 |

OAuth login은 browser를 열고 조직을 선택하도록 요청하며, 갱신 가능한 token을 `~/.config/anthropic/`에 저장합니다. `profile="admin"`으로 client를 생성하면 SDK가 이를 불러옵니다.

대신 Admin API key를 사용하려면 notebook 옆의 `.env` 파일에 `ANTHROPIC_ADMIN_API_KEY=sk-ant-admin01-...`를 추가하세요. setup cell은 key가 설정되어 있으면 이를 사용하고, 없으면 OAuth profile을 사용합니다.

In [1]:
%%capture
%pip install -U "anthropic>=1.1.0" python-dotenv

In [2]:
import os
import time
from datetime import UTC, datetime
from itertools import islice

import anthropic
from dotenv import load_dotenv

load_dotenv()

ADMIN_KEY = os.getenv("ANTHROPIC_ADMIN_API_KEY")
# `ant auth login --profile`에 전달한 profile 이름.
PROFILE = os.getenv("ANTHROPIC_PROFILE", "admin")
# 출력할 rate limit group을 선택하는 데만 사용.
MODEL = "claude-sonnet-5"

if ADMIN_KEY:
    client = anthropic.Anthropic(api_key=ADMIN_KEY)
else:
    client = anthropic.Anthropic(profile=PROFILE)

org = client.beta.organization.retrieve()
auth = "an Admin API key" if ADMIN_KEY else f"OAuth profile {PROFILE!r}"
print(f"Connected to {org.name} ({org.id}) with {auth}")

Connected to Example Org (1f6e8a52-93c4-4d7b-a1e0-5b2c9d8e4f37) with OAuth profile 'admin'


`organization.retrieve()`는 `GET /v1/organizations/me`를 호출합니다. 계속하기 전에 이름을 확인하세요. admin credential은 하나의 조직에 연결되며, 이후 모든 호출이 그 조직을 읽거나 씁니다.

나머지 notebook은 Console에서 알아볼 수 있도록 생성하는 항목에 한 팀의 이름을 사용합니다.

In [3]:
RUN_ID = datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
TEAM = f"research-{RUN_ID}"
print(TEAM)

research-20260826-180659


## 팀원 초대 및 Role 설정

초대는 수신자에게 가입 link를 email로 보내고 수락하면 받게 될 role을 기록합니다. Console 조직에서는 API가 `user`, `developer`, `billing`, `claude_code_user`를 허용합니다. `admin` role은 Console에서만 부여할 수 있습니다.

새 member는 API key 없이 Console과 playground 접근을 허용하는 `user`로 시작하고, 더 필요할 때 승격하세요.

초대는 실제 email을 보냅니다. 일부 조직은 자신의 email domain으로만 초대를 허용하며, 다른 domain으로의 초대는 `400`을 반환합니다. 다음 셀은 어떤 조직에서도 실행되도록 기존 admin의 domain에 있는 placeholder 주소를 기본값으로 사용합니다. 전체 흐름을 시도하려면 `INVITE_EMAIL`을 팀원의 주소로 설정하세요.

In [4]:
INVITE_EMAIL = os.getenv("INVITE_EMAIL")
if not INVITE_EMAIL:
    # 초대 domain을 제한하는 조직에서 다른 domain으로 초대하면 400 반환.
    users = client.beta.organization.users.list(limit=100)
    admin = next((u for u in users if u.role in {"admin", "owner", "primary_owner"}), None)
    if admin is None:
        raise RuntimeError("No admin user to borrow an email domain from. Set INVITE_EMAIL.")
    INVITE_EMAIL = f"new.teammate+{RUN_ID}@{admin.email.split('@', 1)[1]}"

invite = client.beta.organization.invites.create(email=INVITE_EMAIL, role="user")
print(f"{invite.id}  {invite.email}  role={invite.role}  status={invite.status}")
print(f"expires {invite.expires_at:%Y-%m-%d}")

invite_015gWxHNr6h6TdRPZTmuCGnn  new.teammate+20260826-180659@example.com  role=user  status=pending
expires 2026-09-16


초대는 21일 후 만료됩니다. `invites.list()`는 조직의 모든 초대와 상태를 반환하고, `invites.delete()`는 대기 중인 초대를 취소합니다.

In [5]:
for inv in client.beta.organization.invites.list(limit=100):
    print(f"{inv.email:<45} {inv.role:<16} {inv.status:<9} sent {inv.invited_at:%Y-%m-%d}")

new.teammate+20260826-180659@example.com                 user             pending   sent 2026-08-26


누군가 수락하면 `users.list()`에 나타납니다. 다음 셀은 처음 10명을 출력합니다. `email`로 filter하여 한 사람을 찾은 다음 `users.update()`로 role을 바꾸거나 `users.remove()`로 탈퇴 처리합니다. admin은 API로 제거할 수 없습니다.

초대가 대기 중이면 다음 셀은 사용자를 찾지 못해 승격을 건너뛰고, 나중의 membership 셀도 추가를 건너뜁니다. 초대가 수락되면 둘 다 실행됩니다.

In [6]:
# list 호출을 순회하면 모든 페이지를 fetch하므로 출력량 제한.
for user in islice(client.beta.organization.users.list(limit=100), 10):
    print(f"{user.email:<45} {user.role:<16} joined {user.added_at:%Y-%m-%d}")

teammate = next(iter(client.beta.organization.users.list(email=INVITE_EMAIL)), None)
if teammate is None:
    print(f"\n{INVITE_EMAIL} hasn't accepted yet, nothing to promote")
else:
    teammate = client.beta.organization.users.update(teammate.id, role="developer")
    print(f"\n{teammate.email} is now {teammate.role}")

ada@example.com                              admin            joined 2026-06-01



new.teammate+20260826-180659@example.com hasn't accepted yet, nothing to promote


## 팀용 Workspace 생성

workspace는 하나의 조직 안에서 API key, 지출 한도, rate limit, 사용량 보고를 분리합니다. 팀별 또는 환경별 workspace 하나가 일반적인 구성입니다. `tags`에는 cost center 같은 자신의 metadata가 담기며 매번 읽을 때 반환됩니다.

In [7]:
workspace = client.beta.organization.workspaces.create(
    name=TEAM,
    tags={"team": "research", "cost-center": "4711"},
)
print(workspace.id, workspace.name, workspace.tags)

wrkspc_01JwQvzr7rXLA5AGx3HKfFUJ research-20260826-180659 {'team': 'research', 'cost-center': '4711'}


`workspaces.update()`는 이름, 표시 색상, tag를 변경합니다. tag는 기존 항목과 병합되며, tag를 `None`으로 설정하면 제거됩니다.

In [8]:
workspace = client.beta.organization.workspaces.update(
    workspace.id, tags={"env": "sandbox", "cost-center": None}
)
print(workspace.tags)

{'env': 'sandbox', 'team': 'research'}


조직 admin과 billing user는 모든 workspace의 암묵적 member이며 `members.list()`에 나타나지 않습니다. 그 밖의 모든 사람은 `workspace_user`, `workspace_developer`, `workspace_admin` 같은 명시적 workspace role이 필요합니다.

`members.update()`는 role을 바꾸고 `members.remove()`는 조직 membership에 영향을 주지 않고 workspace에서 사람을 제거합니다.

In [9]:
if teammate:
    client.beta.organization.workspaces.members.add(
        workspace.id, user_id=teammate.id, workspace_role="workspace_developer"
    )

members = list(client.beta.organization.workspaces.members.list(workspace.id))
print(f"{workspace.name}: {len(members)} explicit member(s)")
for m in members:
    print(f"  {m.user_id} {m.workspace_role}")

research-20260826-180659: 0 explicit member(s)


## API Key 감사

Admin API는 API key를 나열하고, 이름을 바꾸고, 비활성화하고, archive하지만 생성할 수는 없습니다. 새 key는 Console에서만 만듭니다. provisioning이 아닌 예약된 감사에 사용하세요.

다음 셀은 조직의 모든 활성 key를 순회하며 `MAX_KEY_AGE_DAYS`보다 오래된 key와 만료일이 없는 key를 표시합니다.

In [10]:
MAX_KEY_AGE_DAYS = 90
now = datetime.now(UTC)

# workspaces.list()는 기본 workspace를 반환하지 않으므로 여기 없는 ID는 기본 workspace.
workspace_names = {w.id: w.name for w in client.beta.organization.workspaces.list(limit=1000)}

flagged = []
for key in client.beta.organization.api_keys.list(status="active", limit=1000):
    if key.scope.type == "workspace":
        where = workspace_names.get(key.scope.workspace_id, "Default")
    else:
        where = "(organization)"
    age = (now - key.created_at).days
    expires = f"{key.expires_at:%Y-%m-%d}" if key.expires_at else "never"
    flags = []
    if age > MAX_KEY_AGE_DAYS:
        flags.append("old")
    if key.expires_at is None:
        flags.append("no-expiry")
    if flags:
        flagged.append(key)
    hint = key.partial_key_hint or "-"
    print(f"{key.name:<28} {where:<18} {hint:<24}", end=" ")
    print(f"{age:>4}d  expires {expires:<10} {' '.join(flags)}")

print(f"\n{len(flagged)} key(s) flagged")

backend-prod                 production         sk-ant-api03-R2D...igAA    14d  expires 2027-08-12 
ci-deploy                    production         sk-ant-api03-9xQ...0mAA    14d  expires 2026-12-09 

0 key(s) flagged


key 비활성화는 되돌릴 수 있습니다(`status="active"`로 다시 활성화). archive는 되돌릴 수 없습니다. dry run에는 `APPLY = False`를 유지하고, 표시된 key를 비활성화하려면 `True`로 설정하세요.

먼저 표시된 목록을 읽으세요. 만료일이 없는 key는 production traffic을 처리하더라도 표시되며, 비활성화된 key의 요청은 즉시 실패하기 시작합니다.

In [11]:
APPLY = False

if not flagged:
    print("no flagged keys, nothing to do")
for key in flagged:
    if APPLY:
        key = client.beta.organization.api_keys.update(key.id, status="inactive")
    print(f"{key.name:<28} -> {key.status}{'' if APPLY else '  (dry run, unchanged)'}")

no flagged keys, nothing to do


## Automation용 Service Account 생성

service account는 CI job과 backend service를 위한 조직의 비인간 member입니다. 사람처럼 조직 role과 workspace role을 보유하므로 발급된 credential이 직원 account에 의존하지 않습니다. service account endpoint는 `org:admin` OAuth credential만 허용합니다. Admin API key는 여기서 거부됩니다.

`name`은 소문자, 숫자, hyphen으로 이루어진 slug이며 조직에서 고유해야 합니다. service account endpoint는 생성보다 늦게 반영될 수 있습니다. 몇 초 전에 만든 객체가 이 endpoint에서 잠시 `404`를 반환하거나 목록에 없을 수 있습니다. 방금 만든 항목을 참조하는 호출에서 `404`가 나오면 retry하세요. 다음 셀의 helper가 이를 수행합니다.

In [12]:
# 새 객체는 service account endpoint에서 잠시 404를 반환할 수 있음.
def retry_on_not_found(call, attempts: int = 6, delay: float = 5.0):
    for attempt in range(attempts):
        try:
            return call()
        except anthropic.NotFoundError:
            if attempt == attempts - 1:
                raise
            time.sleep(delay)


service_account = None
if ADMIN_KEY:
    print("Skipped: service account endpoints need the org:admin OAuth credential")
else:
    service_account = client.beta.organization.service_accounts.create(
        name=f"ci-{TEAM}",
        description="Deploy pipeline for the research team",
    )
    print(service_account.id, service_account.name, service_account.organization_role)

svac_01ABCDEFabcdef0123456789XY ci-research-20260826-180659 developer


팀 workspace에 role을 부여한 다음 membership을 나열합니다. 모든 service account는 기본 workspace의 암묵적인 `workspace_user`이기도 하며, 몇 초 전에 부여한 권한이 목록에 나타나기까지 잠시 걸릴 수 있습니다.

In [13]:
if service_account:
    grant = retry_on_not_found(
        lambda: client.beta.organization.service_accounts.workspaces.add(
            service_account.id, workspace_id=workspace.id, workspace_role="workspace_developer"
        )
    )
    print(f"granted {grant.workspace_role} in {workspace.name}\n")
    for m in client.beta.organization.service_accounts.workspaces.list(service_account.id):
        name = workspace.name if m.workspace_id == workspace.id else "Default"
        print(f"{name:<28} {m.workspace_role:<22} {'implicit' if m.implicit else 'explicit'}")

granted workspace_developer in research-20260826-180659

Default                      workspace_user         implicit
research-20260826-180659     workspace_developer    explicit


secret을 저장하지 않고 pipeline이 이 service account로 인증하도록 하려면 CI provider를 federation issuer로 등록하고 OIDC token을 account에 매핑하는 규칙을 추가하세요. [Workload identity federation](https://platform.claude.com/docs/en/manage-claude/workload-identity-federation)이 그 설정을 다룹니다.

## Rate limit 확인

Rate limit은 조직별로 설정됩니다. 각 group은 model family나 `batch`, `files`, `web_search` 같은 API의 다른 부분을 담당합니다. `rate_limits.list()`는 모든 group을 반환합니다. 다음 셀은 setup cell에서 정의한 `MODEL`을 담당하는 group과 model에 연결되지 않은 group을 출력합니다.

In [14]:
for rl in client.beta.organization.rate_limits.list():
    if rl.group_type == "model_group" and MODEL not in (rl.models or []):
        continue
    label = MODEL if rl.group_type == "model_group" else rl.group_type
    limits = ", ".join(f"{limit.type}={limit.value:,}" for limit in rl.limits)
    print(f"{label:<18} {limits}")

batch              enqueued_batch_requests=500,000, requests_per_minute=4,000
claude-sonnet-5    input_tokens_per_minute=10,000,000, output_tokens_per_minute=2,000,000, requests_per_minute=10,000


workspace는 이 값을 상속합니다. admin은 Console에서 workspace의 한도를 더 낮게 설정할 수 있으며, `workspaces.rate_limits.list()`는 override된 값만 반환합니다.

In [15]:
overrides = list(client.beta.organization.workspaces.rate_limits.list(workspace.id))
if not overrides:
    print(f"{workspace.name} has no overrides and inherits the organization limits")
for rl in overrides:
    print(rl.group_type, rl.models, [(limit.type, limit.value) for limit in rl.limits])

research-20260826-180659 has no overrides and inherits the organization limits


## 정리

이 notebook이 생성한 것을 제거합니다. **workspace archive는 영구적이며 그 안의 모든 API key를 취소**하므로 실제 조직에서 실행하기 전에 `workspace.name`을 다시 확인하세요. archive된 workspace는 100-workspace 제한에 포함되지 않습니다.

이전 셀에서 오류가 발생했다면 이 셀을 수동으로 실행하세요. kernel의 `workspace`, `service_account`, `invite`가 필요합니다.

In [16]:
archived = client.beta.organization.workspaces.archive(workspace.id)
print(f"archived workspace {archived.name} at {archived.archived_at:%Y-%m-%d %H:%M}")

if service_account:
    retry_on_not_found(
        lambda: client.beta.organization.service_accounts.archive(service_account.id)
    )
    print(f"archived service account {service_account.name}")

# 수락된 초대를 삭제하면 해당 사람이 이제 member이므로 400 반환.
if client.beta.organization.invites.retrieve(invite.id).status == "pending":
    client.beta.organization.invites.delete(invite.id)
    print(f"deleted invite for {invite.email}")
else:
    print(f"{invite.email} joined the organization, offboard them with users.remove()")

archived workspace research-20260826-180659 at 2026-08-26 18:07
archived service account ci-research-20260826-180659
deleted invite for new.teammate+20260826-180659@example.com


## 다음 단계

모두 `client.beta.organization`을 통해 admin으로 인증하고, 사용자를 초대하고, workspace를 만들고, service account에 접근 권한을 주고, 조직 API key를 감사하고, rate limit을 읽었습니다.

여기서 다루지 않은 관련 admin 기능:

- [사용량 및 비용 cookbook](../observability/usage_cost_api.ipynb)과 [Usage and Cost API](https://platform.claude.com/docs/en/manage-claude/usage-cost-api): workspace, key, model별 token 및 달러 보고
- [Claude Code Analytics API](https://platform.claude.com/docs/en/manage-claude/claude-code-analytics-api): 조직의 사용자별 Claude Code 활동
- [Workload identity federation](https://platform.claude.com/docs/en/manage-claude/workload-identity-federation): `client.beta.organization.federation`을 통한 생성한 service account의 key 없는 인증
- [Admin API reference](https://platform.claude.com/docs/en/api/admin): SDK가 `external_keys`와 `federation` 아래 제공하는 것을 포함한 모든 endpoint